# 03 · team_success explore
goal: build `won_league` for `team_success` (league champion per season, Big 5, 2010–2025).

## 1. setup

In [1]:
import pandas as pd
import requests
from io import StringIO

headers = {"User-Agent": "Mozilla/5.0"}


## 2. Premier League from Wikipedia
read the table → clean columns → remove footnotes and separator rows → convert season to BdO format

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_English_football_champions"
response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text), match="Season")

len(tables)

1

In [3]:
champs = tables[0]

In [4]:
champs.columns = champs.columns.droplevel(1)

In [5]:
champs = champs[["Season", "Champions (number of titles)"]]
champs["Champions (number of titles)"] = champs["Champions (number of titles)"].str.replace(r"\[.*?\]|\(\d+\)", "", regex=True).str.strip()
champs

/var/folders/8g/3fq_9hs93q32xgd2y59z1bcw0000gn/T/ipykernel_38988/1437591506.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  champs["Champions (number of titles)"] = champs["Champions (number of titles)"].str.replace(r"\[.*?\]|\(\d+\)", "", regex=True).str.strip()


,Season,Champions (number of titles)
0,1888–89,Preston North End
1,1889–90,Preston North End
2,1890–91,Everton
3,1891–92,Sunderland
4,Football League First Division (1892–1992),Football League First Division (1892–1992)
...,...,...
127,2022–23,Manchester City
128,2023–24,Manchester City
129,2024–25,Liverpool
130,2025–26,Arsenal


In [6]:
champs = champs[champs["Season"].str.match(r"^\d{4}–\d{2}$", na=False)]

In [7]:
champs["bdo_season"] = champs["Season"].str[:4].astype(int) + 1

/var/folders/8g/3fq_9hs93q32xgd2y59z1bcw0000gn/T/ipykernel_38988/911476969.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  champs["bdo_season"] = champs["Season"].str[:4].astype(int) + 1


In [8]:
champs = champs[champs["bdo_season"].between(2010, 2025)]

champs

,Season,Champions (number of titles),bdo_season
114,2009–10,Chelsea,2010
115,2010–11,Manchester United,2011
116,2011–12,Manchester City,2012
117,2012–13,Manchester United,2013
118,2013–14,Manchester City,2014
119,2014–15,Chelsea,2015
120,2015–16,Leicester City,2016
121,2016–17,Chelsea,2017
122,2017–18,Manchester City,2018
123,2018–19,Manchester City,2019


In [9]:
champs = champs.drop(columns=["Season"])

In [10]:
champs = champs.rename(columns={"Champions (number of titles)": "champions", "bdo_season": "season"})
champs["league"] = "ENG-Premier League"
# ⚠️ don't run again: it overwrites the CSV with only the Premier League (the other 4 leagues were added by hand)
# champs.to_csv("../data/manual/league_champions.csv", index=False)

## 3. check: 16 rows per league (the other 4 leagues were filled from Wikipedia by an automating bot)

In [11]:
pd.read_csv("../data/manual/league_champions.csv").league.value_counts()

league
ENG-Premier League    16
ESP-La Liga           16
FRA-Ligue 1           16
GER-Bundesliga        16
ITA-Serie A           16
Name: count, dtype: int64

## 4. club names: champions ↔ nominees
`nominees.main_club` is the reference. what's left in the set are clubs with no nominees (Lille, Marseille, Montpellier).

In [59]:
teams = pd.read_csv("../data/manual/team_success.csv")
champs = pd.read_csv("../data/manual/league_champions.csv")

champ_map = {"Inter": "Internazionale", "Dortmund": "Borussia Dortmund", "Leverkusen": "Bayer Leverkusen"}
champs["champions"] = champs["champions"].replace(champ_map)

set(champs.champions) - set(teams.main_club)

{'Lille', 'Marseille', 'Montpellier'}

In [13]:
teams[teams.main_club.str.contains("Dortmund|Inter|Leverkusen")].main_club.unique()

array(['Internazionale', 'Borussia Dortmund', 'Inter Miami',
       'Bayer Leverkusen'], dtype=object)

> Inter Milan → Internazionale is fixed in `wiki_reader.py`, not here.

In [14]:
pd.read_csv("../data/manual/team_success.csv").shape

(189, 2)

## 5. Add won_league column

In [15]:
# pattern
merged = teams.merge(champs, left_on=["season","main_club"], right_on=["season","champions"], how="left")
merged["won_league"] = merged["champions"].notna().astype(int)

In [16]:
merged = merged.drop(columns=["champions", "league"])

In [17]:
set(champs.champions) - set(merged[merged.won_league == 1].main_club)

{'Borussia Dortmund', 'Lille', 'Marseille', 'Montpellier'}

In [19]:
merged.won_league.sum()

np.int64(67)

In [20]:
merged.to_csv("../data/manual/team_success.csv", index=False)

## 6. CL winner and runners-up

In [40]:
url_cl = "https://en.wikipedia.org/wiki/List_of_European_Cup_and_UEFA_Champions_League_finals"
response = requests.get(url_cl, headers=headers)
tables = pd.read_html(StringIO(response.text), match="Winners")

champs_cl = tables[0]
champs_cl

,Season,Country,Winners,Score,Runners-up,Country.1,Venue,Attendance[15]
0,1955–56,Spain,Real Madrid,4–3,Reims,France,"Parc des Princes, Paris, France",38239
1,1956–57,Spain,Real Madrid,2–0,Fiorentina,Italy,"Santiago Bernabéu, Madrid, Spain",124000
2,1957–58,Spain,Real Madrid,3–2†,Milan,Italy,"Heysel Stadium, Brussels, Belgium",67000
3,1958–59,Spain,Real Madrid,2–0,Reims,France,"Neckarstadion, Stuttgart, West Germany",72000
4,1959–60,Spain,Real Madrid,7–3,Eintracht Frankfurt,West Germany,"Hampden Park, Glasgow, Scotland",127621
...,...,...,...,...,...,...,...,...
67,2021–22,Spain,Real Madrid,1–0,Liverpool,England,"Stade de France, Saint-Denis, France",75000
68,2022–23,England,Manchester City,1–0,Inter Milan,Italy,"Atatürk Olympic Stadium, Istanbul, Turkey",71412
69,2023–24,Spain,Real Madrid,2–0,Borussia Dortmund,Germany,"Wembley Stadium, London, England",86212
70,2024–25,France,Paris Saint-Germain,5–0,Inter Milan,Italy,"Allianz Arena, Munich, Germany",64327


In [41]:
champs_cl.tail(17)

,Season,Country,Winners,Score,Runners-up,Country.1,Venue,Attendance[15]
55,2009–10,Italy,Inter Milan,2–0,Bayern Munich,Germany,"Santiago Bernabéu, Madrid, Spain",73490
56,2010–11,Spain,Barcelona,3–1,Manchester United,England,"Wembley Stadium, London, England",87695
57,2011–12,England,Chelsea,1–1*[j],Bayern Munich,Germany,"Allianz Arena, Munich, Germany",62500
58,2012–13,Germany,Bayern Munich,2–1,Borussia Dortmund,Germany,"Wembley Stadium, London, England",86298
59,2013–14,Spain,Real Madrid,4–1†,Atlético Madrid,Spain,"Estádio da Luz, Lisbon, Portugal",60976
60,2014–15,Spain,Barcelona,3–1,Juventus,Italy,"Olympiastadion, Berlin, Germany",70442
61,2015–16,Spain,Real Madrid,1–1*[k],Atlético Madrid,Spain,"San Siro, Milan, Italy",71942
62,2016–17,Spain,Real Madrid,4–1,Juventus,Italy,"Millennium Stadium, Cardiff, Wales",65842
63,2017–18,Spain,Real Madrid,3–1,Liverpool,England,"NSC Olimpiyskiy Stadium, Kyiv, Ukraine",61561
64,2018–19,England,Liverpool,2–0,Tottenham Hotspur,England,"Metropolitano Stadium, Madrid, Spain",63272


In [42]:
champs_cl = champs_cl[["Season", "Winners", "Runners-up"]].copy()
champs_cl.tail(17)


,Season,Winners,Runners-up
55,2009–10,Inter Milan,Bayern Munich
56,2010–11,Barcelona,Manchester United
57,2011–12,Chelsea,Bayern Munich
58,2012–13,Bayern Munich,Borussia Dortmund
59,2013–14,Real Madrid,Atlético Madrid
60,2014–15,Barcelona,Juventus
61,2015–16,Real Madrid,Atlético Madrid
62,2016–17,Real Madrid,Juventus
63,2017–18,Real Madrid,Liverpool
64,2018–19,Liverpool,Tottenham Hotspur


In [ ]:
#champs_cl.Season = champs_cl.Season.str[:4].astype(int) + 1
champs_cl = champs_cl[champs_cl.Season.between(2010, 2025)]

champs_cl

,Season,Winners,Runners-up
55,2010,Inter Milan,Bayern Munich
56,2011,Barcelona,Manchester United
57,2012,Chelsea,Bayern Munich
58,2013,Bayern Munich,Borussia Dortmund
59,2014,Real Madrid,Atlético Madrid
60,2015,Barcelona,Juventus
61,2016,Real Madrid,Atlético Madrid
62,2017,Real Madrid,Juventus
63,2018,Real Madrid,Liverpool
64,2019,Liverpool,Tottenham Hotspur


In [ ]:
club_map = {"Inter Milan": "Internazionale"}
champs_cl["Winners"] = champs_cl["Winners"].replace(club_map)
champs_cl["Runners-up"] = champs_cl["Runners-up"].replace(club_map)
champs_cl

/var/folders/8g/3fq_9hs93q32xgd2y59z1bcw0000gn/T/ipykernel_38988/358005252.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  champs_cl["Winners"] = champs_cl["Winners"].replace(club_map)
/var/folders/8g/3fq_9hs93q32xgd2y59z1bcw0000gn/T/ipykernel_38988/358005252.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  champs_cl["Runners-up"] = champs_cl["Runners-up"].replace(club_map)


,Season,Winners,Runners-up
55,2010,Internazionale,Bayern Munich
56,2011,Barcelona,Manchester United
57,2012,Chelsea,Bayern Munich
58,2013,Bayern Munich,Borussia Dortmund
59,2014,Real Madrid,Atlético Madrid
60,2015,Barcelona,Juventus
61,2016,Real Madrid,Atlético Madrid
62,2017,Real Madrid,Juventus
63,2018,Real Madrid,Liverpool
64,2019,Liverpool,Tottenham Hotspur


In [47]:
winners = champs_cl[["Season", "Winners"]].copy()
winners.rename(columns={"Season": "season", "Winners": "main_club"}, inplace=True)
winners['ucl_stage'] = 6

winners

,season,main_club,ucl_stage
55,2010,Internazionale,6
56,2011,Barcelona,6
57,2012,Chelsea,6
58,2013,Bayern Munich,6
59,2014,Real Madrid,6
60,2015,Barcelona,6
61,2016,Real Madrid,6
62,2017,Real Madrid,6
63,2018,Real Madrid,6
64,2019,Liverpool,6


In [48]:
runners_up = champs_cl[["Season", "Runners-up"]].copy()
runners_up.rename(columns={"Season": "season", "Runners-up": "main_club"}, inplace=True)   
runners_up['ucl_stage'] = 5

runners_up

,season,main_club,ucl_stage
55,2010,Bayern Munich,5
56,2011,Manchester United,5
57,2012,Bayern Munich,5
58,2013,Borussia Dortmund,5
59,2014,Atlético Madrid,5
60,2015,Juventus,5
61,2016,Atlético Madrid,5
62,2017,Juventus,5
63,2018,Liverpool,5
64,2019,Tottenham Hotspur,5


In [50]:
ucl = pd.concat([winners, runners_up], ignore_index=True)

ucl

,season,main_club,ucl_stage
0,2010,Internazionale,6
1,2011,Barcelona,6
2,2012,Chelsea,6
3,2013,Bayern Munich,6
4,2014,Real Madrid,6
5,2015,Barcelona,6
6,2016,Real Madrid,6
7,2017,Real Madrid,6
8,2018,Real Madrid,6
9,2019,Liverpool,6


In [52]:
ucl.to_csv("../data/manual/ucl_stages.csv", index=False)